## Étape 2 : chargement de la base opérationnelle

Ce notebook charge les 4 fichiers Excel du stage dans des tables Snowflake :
- `PRICELINES_BAF09` : extraction complète, dédoublonnée
- `PRICELINES_BAF09_UPDATE` : seconde photo
- `REF_CAR_CHARGES` : codes de charges
- `REF_GOL_SERVICES` : lignes maritimes et services

Fichiers sources dans `@PROJECT_DB.PUBLIC.FILES/stage_1/`.

In [ ]:
%pip install openpyxl

In [ ]:
import io
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
STAGE = "@PROJECT_DB.PUBLIC.FILES"

### 1. Référentiels légers (CAR, GOL)

In [ ]:
with session.file.get_stream(f"{STAGE}/stage_1/CAR surcharges.xlsx") as stream:
    car_df = pd.read_excel(io.BytesIO(stream.read()))
print("CAR surcharges :", car_df.shape)
print(car_df.columns.tolist())
car_df.head(5)

In [ ]:
with session.file.get_stream(f"{STAGE}/stage_1/GOL Referential.xlsx") as stream:
    gol_df = pd.read_excel(io.BytesIO(stream.read()))
print("GOL Referential :", gol_df.shape)
print(gol_df.columns.tolist())
gol_df.head(5)

### 2. Extraction BAF09 (fichier principal, ~1 M lignes)

In [ ]:
with session.file.get_stream(f"{STAGE}/stage_1/Extraction BAF09.xlsx") as stream:
    baf_raw = pd.read_excel(io.BytesIO(stream.read()), sheet_name='BAF09 full extract')
print("Shape brute :", baf_raw.shape)
print("Colonnes :", baf_raw.columns.tolist())
baf_raw.head(3)

In [ ]:
print("Types :")
print(baf_raw.dtypes)
print("\nValeurs uniques de charge_origin :", baf_raw['charge_origin'].unique())
print("Valeurs uniques de pricelist_line_type :", baf_raw['pricelist_line_type'].unique())
print("Valeurs uniques de applicable :", baf_raw['applicable'].unique())
print("Valeurs uniques de fixed :", baf_raw['fixed'].unique())
print("Valeurs uniques de rate_structure :", baf_raw['rate_structure'].unique())

In [ ]:
n_before = len(baf_raw)
dupes = baf_raw.duplicated(keep='first')
n_dupes = dupes.sum()
print(f"Lignes brutes : {n_before}")
print(f"Doublons exacts : {n_dupes}")
print(f"Après suppression : {n_before - n_dupes}")

assert n_dupes == 459, f"Attendu 459 doublons, trouvé {n_dupes}"

In [ ]:
baf = baf_raw[~dupes].copy()

baf['last_modification_date'] = pd.to_datetime(baf['last_modification_date'], format='mixed')

baf.columns = [c.upper() for c in baf.columns]

key_dupes = baf.duplicated(subset=['DETAIL_UID', 'CHARGE_ORIGIN'], keep=False)
print(f"Lignes partageant la même clé (detail_uid, charge_origin) : {key_dupes.sum()}")
print(f"Clés distinctes (detail_uid, charge_origin) : {baf[['DETAIL_UID','CHARGE_ORIGIN']].drop_duplicates().shape[0]}")
print(f"Lignes finales : {len(baf)}")
print(f"\nlast_modification_date dtype : {baf['LAST_MODIFICATION_DATE'].dtype}")
baf.head(3)

### 3. Update BAF09

In [ ]:
with session.file.get_stream(f"{STAGE}/stage_1/Update BAF09.xlsx") as stream:
    upd_bytes = stream.read()

xl_upd = pd.ExcelFile(io.BytesIO(upd_bytes))
print("Feuilles :", xl_upd.sheet_names)

data_sheet = xl_upd.sheet_names[-1]
upd_raw = xl_upd.parse(data_sheet)
print(f"Sheet '{data_sheet}' : {upd_raw.shape}")
print("Colonnes :", upd_raw.columns.tolist())

upd = upd_raw.copy()

# Renommer les colonnes pour aligner avec l'extraction principale
upd = upd.rename(columns={
    'modified_date': 'last_modification_date',
    'rate_1': 'rate_20ST',
    'rate_2': 'rate_40ST',
    'rate_3': 'rate_40HC',
    'rate_4': 'rate_45HC',
})

upd['last_modification_date'] = pd.to_datetime(upd['last_modification_date'], format='mixed')
upd.columns = [c.upper() for c in upd.columns]

print(f"\nUpdate BAF09 prêt : {len(upd)} lignes")
print("Colonnes finales :", upd.columns.tolist())
upd.head(3)

### 4. Écriture des 4 tables

In [ ]:
# Préparer CAR
car = car_df.copy()
car.columns = [c.upper().replace(' ', '_').replace('/', '_') for c in car.columns]

# Préparer GOL (forcer tout en texte pour éviter les types mixtes)
gol = gol_df.copy()
gol.columns = [c.upper() for c in gol.columns]
for col in gol.columns:
    gol[col] = gol[col].astype(str)

# Forcer les colonnes object en str pour BAF (LINE_SEQUENCE a des types mixtes)
for df in [baf, upd]:
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str)
    df.reset_index(drop=True, inplace=True)

tables = {
    'REF_CAR_CHARGES': car,
    'REF_GOL_SERVICES': gol,
    'PRICELINES_BAF09': baf,
    'PRICELINES_BAF09_UPDATE': upd,
}

for name, data in tables.items():
    session.write_pandas(
        data, name,
        database='PROJECT_DB', schema='PUBLIC',
        auto_create_table=True, overwrite=True,
    )
    print(f"{name} : {len(data)} lignes")

### 5. Vérification

In [ ]:
%%sql -r table_counts
SELECT 'PRICELINES_BAF09' AS tbl, COUNT(*) AS row_count FROM PROJECT_DB.PUBLIC.PRICELINES_BAF09
UNION ALL
SELECT 'PRICELINES_BAF09_UPDATE', COUNT(*) FROM PROJECT_DB.PUBLIC.PRICELINES_BAF09_UPDATE
UNION ALL
SELECT 'REF_CAR_CHARGES', COUNT(*) FROM PROJECT_DB.PUBLIC.REF_CAR_CHARGES
UNION ALL
SELECT 'REF_GOL_SERVICES', COUNT(*) FROM PROJECT_DB.PUBLIC.REF_GOL_SERVICES

In [ ]:
%%sql -r key_check
-- Test clé composite : chaque (DETAIL_UID, CHARGE_ORIGIN) doit être unique
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT DETAIL_UID || '|' || CHARGE_ORIGIN) AS distinct_keys
FROM PROJECT_DB.PUBLIC.PRICELINES_BAF09

### 6. Tests d'acceptation de `find_pricelines`

In [ ]:
import os, sys, importlib
parent = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent not in sys.path:
    sys.path.insert(0, parent)

import src.tools.pricelines
importlib.reload(src.tools.pricelines)
from src.tools.pricelines import find_pricelines
print('find_pricelines importé')

In [ ]:
# Test 1 : MXVER-ESSCT, 2026-04-10, ST
# Attendu : detail_uid 360146852 avec 2 versions (CTB + Manual), evidence=True
r1 = find_pricelines(session, 'MXVER', 'ESSCT', '2026-04-10', container_type='ST')

uid_360 = [l for l in r1['lines'] if l['detail_uid'] == 360146852]
assert len(uid_360) == 2, f"detail_uid 360146852 devrait avoir 2 versions, trouvé {len(uid_360)}"
assert {l['charge_origin'] for l in uid_360} == {'CTB', 'Manual'}
assert r1['evidence'] is True
print(f"Test 1 OK : {r1['nb_candidates']} candidate(s), evidence={r1['evidence']}")
print(f"Finding : {r1['finding']}")

In [ ]:
# Test 2 : BDCGP-MAAGA, 2026-03-18, ST
# Attendu : 1 candidate (detail_uid 359468845), 1 version, evidence=False
r2 = find_pricelines(session, 'BDCGP', 'MAAGA', '2026-03-18', container_type='ST')

assert r2['nb_candidates'] == 1, f"Attendu 1 candidate, trouvé {r2['nb_candidates']}"
assert r2['lines'][0]['detail_uid'] == 359468845
assert r2['evidence'] is False
print(f"Test 2 OK : {r2['nb_candidates']} candidate(s), evidence={r2['evidence']}")

In [ ]:
# Test 3 : AEJEA-AUADL, 2026-03-20, sans type de conteneur
# Attendu : 2 candidates (lignes 24804-2 en ST et 24333-4 en RF)
r3 = find_pricelines(session, 'AEJEA', 'AUADL', '2026-03-20')

assert r3['nb_candidates'] == 2, f"Attendu 2 candidates, trouvé {r3['nb_candidates']}"
seqs = {l['line_sequence'] for l in r3['lines']}
assert '24804-2' in seqs and '24333-4' in seqs, f"Séquences attendues non trouvées : {seqs}"
types = {l['pricelist_line_type'] for l in r3['lines']}
assert types == {'ST', 'RF'}, f"Types attendus ST et RF, trouvé {types}"
print(f"Test 3 OK : {r3['nb_candidates']} candidate(s), types={types}")

In [ ]:
# Test 4 : route inexistante
r4 = find_pricelines(session, 'ZZZZZ', 'XXXXX', '2026-01-01')
assert r4['lines'] == []
assert r4['nb_candidates'] == 0
assert r4['evidence'] is False
print('Test 4 OK : résultat vide, pas d\'erreur')

print('\n=== Les 4 tests d\'acceptation passent ===')